In [1]:
import os
import requests
from exa_py import Exa
from dotenv import load_dotenv
from openai import OpenAI
import json
from datetime import datetime, timedelta

In [2]:
load_dotenv()

openweather_api_key = os.getenv("OPENWEATHER_API_KEY")

exa_api_key = os.getenv("EXA_API_KEY")

exa = Exa(api_key=exa_api_key)

client = OpenAI()

In [3]:
def get_weather(lat, lon):
    """
    Fetches weather data and returns a cleaned dictionary 
    optimized for an LLM Agent's context.
    """
    # Use 'exclude' to save bandwidth and keep the response small
    exclude = "minutely,hourly"
    url = f"https://api.openweathermap.org/data/3.0/onecall?lat={lat}&lon={lon}&exclude={exclude}&appid={openweather_api_key}&units=metric"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # 2. Focus on Semantic Fields
        # We extract only what an agent needs to hold a conversation
        current = data.get("current", {})
        daily_today = data.get("daily", [{}])[0]

        weather_summary = {
            "location_coords": {"lat": lat, "lon": lon},
            "current_condition": current.get("weather", [{}])[0].get("description"),
            "temperature": {
                "actual": f"{current.get('temp')}°C",
                "feels_like": f"{current.get('feels_like')}°C"
            },
            "humidity": f"{current.get('humidity')}%",
            "uv_index": current.get("uvi"),
            "today_forecast": {
                "summary": daily_today.get("summary"), # One Call 3.0 specific human-readable string
                "pop_chance": f"{int(daily_today.get('pop', 0) * 100)}%", # Prob. of precipitation
                "temp_range": f"{daily_today.get('temp', {}).get('min')}°C to {daily_today.get('temp', {}).get('max')}°C"
            },
            "alerts": data.get("alerts", "No active weather alerts")
        }

        return weather_summary

    except Exception as e:
        return {"error": f"Failed to fetch weather: {str(e)}"}

# --- Experimenting with Lagos ---
#OMU_ARAN_LAT = 8.1386
#OMU_ARAN_LON = 5.1026

#weather_data = get_weather(OMU_ARAN_LAT, OMU_ARAN_LON)
#print(weather_data)


In [4]:
def exa_search(query: str):
    # 1. Fetch minimal results
    response = exa.search(
        query=query, 
        type='auto', 
        num_results=2,  # Drop to 2 results for maximum savings
        start_published_date=(datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d'),
        contents={"text": {"max_characters": 300}} # Drop to 300 chars
    )
    
    # 2. MANUALLY extract only what the LLM needs
    # This turns a massive object into a tiny, clean list of dicts
    cleaned_results = []
    for r in response.results:
        cleaned_results.append({
            "title": r.title,
            "url": r.url,
            "content": r.text  # Only the text snippet
        })
        
    return cleaned_results

In [5]:
tools = [
    {
    "type": "function",
    "name": "get_weather",
    "description": "Get current weather conditions and daily forecast for a specific location using latitude and longitude coordinates.",
    "parameters": {
        "type": "object",
        "properties": {
            "lat": {
                "type": "number",
                "description": "The latitude of the location (e.g., 6.4550 for Lagos).",
                "minimum": -90,
                "maximum": 90
            },
            "lon": {
                "type": "number",
                "description": "The longitude of the location (e.g., 3.3841 for Lagos).",
                "minimum": -180,
                "maximum": 180
            },
        },
        "required": ["lat", "lon"],
        "additionalProperties": False
    },
},

    {
        "type": "function",
        "name": "exa_search",
        "description": "Perform a search query on the web, and retrieve the most relevant URLs/web data.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to perform.",
                },
            },
            "required": ["query"],
        },
    },

]

In [6]:
AVAILABLE_TOOLS = {
    "exa_search": exa_search,
    "get_weather": get_weather,
}


In [15]:
def execute_tool_calls(tool_call):
    f_name = tool_call.name
    f_args = json.loads(tool_call.arguments)

    # 1. Look up the function in the registry
    function_to_call = AVAILABLE_TOOLS.get(f_name)

    if function_to_call:
        # 2. Execute it dynamically
        results = function_to_call(**f_args)
        
        # 3. Return the standard wrapper
        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(results)
        }
    
    raise ValueError(f"The model tried to call '{f_name}', but it isn't in AVAILABLE_TOOLS.")


In [16]:
SYSTEM_MESSAGE = {
    "role": "system",
    "content": (
        "You are a versatile research assistant with access to real-time tools.\n\n"
        "1. EXA SEARCH: Use this for web-based queries. Every claim based on search "
        "results MUST cite its source URL directly. If results are insufficient, state that.\n"
        "2. WEATHER DATA: Use this for current conditions or forecasts using coordinates.\n\n"
        "Always choose the most appropriate tool for the user's request. "
        "You may use multiple tools in sequence if needed (e.g., searching for a "
        "city's coordinates before checking its weather)."
    ),
}


In [17]:
def log_step(step, tool_calls):
    print(f"\n{'='*15} ITERATION {step} {'='*15}")
    print(f"Tool Calls this turn: {len(tool_calls)}")
    for i, tool in enumerate(tool_calls, 1):
        print(f"  [{i}] Executing: {tool.name}")
        print(f"      Arguments: {tool.arguments}")


In [18]:
import json

def print_conversation(conversation):
    print("\n" + "="*20 + " CONVERSATION HISTORY " + "="*20)
    for msg in conversation:
        # --- Case 1: Standard Dictionaries (User/System/Tool Results) ---
        if isinstance(msg, dict):
            # Safe way to get role without crashing
            role = msg.get("role", msg.get("type", "TOOL")).upper()
            
            # Print text content (User/System)
            if "content" in msg and isinstance(msg["content"], str):
                print(f"\n[{role}]: {msg['content']}")
            
            # Print Tool Results (formatted JSON)
            elif msg.get("type") == "function_call_output":
                print(f"\n[TOOL RESULT - ID: {msg.get('call_id', 'unknown')[:8]}...]:")
                try:
                    data = json.loads(msg["output"])
                    print(json.dumps(data, indent=4))
                except:
                    print(msg["output"])

        # --- Case 2: SDK Objects (Assistant/Tool Calls) ---
        else:
            # Handle the ResponseFunctionToolCall objects
            if hasattr(msg, 'type') and msg.type == "function_call":
                print(f"\n[ASSISTANT CALLING TOOL]: {msg.name}")
                print(f"Arguments: {msg.arguments}")
            
            # Handle the final Text Response objects
            elif hasattr(msg, 'type') and msg.type == "message":
                # Assuming response.output_text was used in your logic
                # We pull the actual text from the content list if needed
                print(f"\n[ASSISTANT]: (Final Response Provided)")

    print("\n" + "="*20 + " END OF HISTORY " + "="*20 + "\n")

In [22]:
MAX_MESSAGES = 10

def run_agent(user_query, conversation):
    conversation.append({"role": "user", "content": user_query})
    
    # SLIDING WINDOW LOGIC
    while len(conversation) > MAX_MESSAGES:
        if len(conversation) > 1:
            conversation.pop(1)
            
    iteration_count = 0
    total_tools_executed = 0

    while True:
        iteration_count += 1
        response = client.responses.create(
            model="gpt-4o-mini",
            tools=tools,
            input=conversation,
        )

        # Append model output to conversation
        conversation += response.output

        # Collect tool calls
        tool_calls = [
            item for item in response.output 
            if item.type == "function_call"
        ]

        # NEW: Print the formatted log for this turn
        log_step(iteration_count, tool_calls)
        total_tools_executed += len(tool_calls)
        

        if not tool_calls:
            # Print Final Summary before breaking
            print(f"\n{'*'*10} AGENT FINISHED {'*'*10}")
            print(f"Total Iterations: {iteration_count}")
            print(f"Total Tool Calls: {total_tools_executed}")
            print(f"{'*'*36}\n")
            print(response.output_text)
            print_conversation(conversation)
            print(conversation)
            break

        # Execute each tool call
        for tool_call in tool_calls:
            tool_result = execute_tool_calls(tool_call)
            conversation.append(tool_result)
            

In [4]:
def main():
    # 1. LOAD: Fetch old memories and inject into System Message
    memories = load_long_term_memory()
    SYSTEM_MESSAGE["content"] += memories
    
    # 2. INITIALIZE: The session history (Short-Term Memory)
    session_history = [SYSTEM_MESSAGE]
    
    print("--- AI Research Agent (with LTM) Active ---")
    
    while True:
        try:
            user_query = input("\nWhat do you want to search for? (type 'exit' to stop): ")
            
            if user_query.lower() in ["exit", "quit"]:
                # 3. SAVE: Before quitting, summarize the session
                print("Reflecting on our conversation...")
                extract_and_save_memories(session_history)
                print("Goodbye!")
                break
                
            if not user_query.strip():
                continue
                
            # Pass the persistent session_history for Short-Term Memory
            run_agent(user_query, session_history)
            
        except Exception as e:
            print(f"Error: {e}")


In [21]:
main()

--- AI Agent with Memory Active ---


What do you want to search for? (type 'exit' to stop):  What's the weather in Paris



=============== ITERATION 1 ===============
Tool Calls this turn: 1
  [1] Executing: get_weather
      Arguments: {"lat":48.8566,"lon":2.3522}

=============== ITERATION 2 ===============
Tool Calls this turn: 0

********** AGENT FINISHED **********
Total Iterations: 2
Total Tool Calls: 1
************************************

The current weather in Paris is:

- **Condition:** Clear sky
- **Temperature:** 13.6°C (feels like 12.61°C)
- **Humidity:** 61%
- **UV Index:** 0

### Today's Forecast:
- **Summary:** Expect a day that is partly cloudy with clear spells.
- **Temperature Range:** 6.43°C to 16.77°C
- **Chance of Precipitation:** 0%

There are no active weather alerts at the moment.

==================== CONVERSATION HISTORY ====================

[SYSTEM]: You are a versatile research assistant with access to real-time tools.

1. EXA SEARCH: Use this for web-based queries. Every claim based on search results MUST cite its source URL directly. If results are insufficient, state that.

What do you want to search for? (type 'exit' to stop):  what of the food there



=============== ITERATION 1 ===============
Tool Calls this turn: 1
  [1] Executing: exa_search
      Arguments: {"query":"best food to try in Paris"}

=============== ITERATION 2 ===============
Tool Calls this turn: 0

********** AGENT FINISHED **********
Total Iterations: 2
Total Tool Calls: 1
************************************

Here are some recommendations for traditional foods to try in Paris:

1. **Croissants** - Flaky, buttery pastries perfect for breakfast.
2. **Escargot** - Snails cooked in garlic butter; a French delicacy.
3. **Coq au Vin** - Chicken braised with wine, mushrooms, onions, and bacon.
4. **Ratatouille** - A vegetable dish made with zucchini, eggplant, and tomatoes.
5. **Crepes** - Thin pancakes that can be filled with a variety of sweet or savory ingredients.
6. **Macarons** - Colorful, delicate meringue cookies with various fillings.
7. **Boeuf Bourguignon** - A hearty beef stew cooked slowly in red wine.

For a more detailed exploration of these dishes, yo

What do you want to search for? (type 'exit' to stop):  exit


Session history so far


[{'role': 'system', 'content': "You are a versatile research assistant with access to real-time tools.\n\n1. EXA SEARCH: Use this for web-based queries. Every claim based on search results MUST cite its source URL directly. If results are insufficient, state that.\n2. WEATHER DATA: Use this for current conditions or forecasts using coordinates.\n\nAlways choose the most appropriate tool for the user's request. You may use multiple tools in sequence if needed (e.g., searching for a city's coordinates before checking its weather)."}, {'role': 'user', 'content': "What's the weather in Paris"}, ResponseFunctionToolCall(arguments='{"lat":48.8566,"lon":2.3522}', call_id='call_Y8FHWbzFKsnECqlpVSQd3XFU', name='get_weather', type='function_call', id='fc_03c5637cc69ee4040069a72c7fccf881909ba169d30a6e19fa', status='completed'), {'type': 'function_call_output', 'call_id': 'call_Y8FHWbzFKsnECqlpVSQd3XFU', 'output': '{"location_coords": {"lat": 48.8566, "lon": 2.3522}, "curr

In [1]:
import json
import os

MEMORY_FILE = "memory.json"

def load_long_term_memory():
    """Reads the JSON vault and returns a formatted string for the System Message."""
    if not os.path.exists(MEMORY_FILE):
        return ""
    
    try:
        with open(MEMORY_FILE, "r") as f:
            memories = json.load(f)
        
        if not memories:
            return ""
            
        formatted_memories = "\n\nRELEVANT FACTS FROM PREVIOUS SESSIONS:\n"
        for item in memories:
            formatted_memories += f"- {item['fact']} (Category: {item.get('category', 'General')})\n"
        return formatted_memories
    except Exception as e:
        print(f"Error loading memory: {e}")
        return ""

def save_memory_notes(new_facts):
    """Appends new facts to the JSON vault."""
    if not new_facts:
        return

    existing_memories = []
    if os.path.exists(MEMORY_FILE):
        try:
            with open(MEMORY_FILE, "r") as f:
                existing_memories = json.load(f)
        except:
            existing_memories = []

    existing_memories.extend(new_facts)

    with open(MEMORY_FILE, "w") as f:
        json.dump(existing_memories, f, indent=4)
    print(f"--- Saved {len(new_facts)} new facts to Long-Term Memory ---")


In [3]:
def extract_and_save_memories(conversation):
    """Hidden turn to extract permanent facts from the chat history."""
    extraction_prompt = (
        "Analyze the conversation above. Extract only PERMANENT facts about the user "
        "(e.g., their location, name, or specific preferences). Ignore temporary data "
        "like current weather numbers. Output ONLY a valid JSON list of objects "
        "with 'fact' and 'category' keys. Example: [{'fact': 'User lives in Lagos', 'category': 'location'}]. "
        "If no new facts, return []."
    )
    
    # Use your verbatim client.responses.create pattern
    response = client.responses.create(
        model="gpt-4o-mini",
        input=conversation + [{"role": "user", "content": extraction_prompt}]
    )
    
    try:
        # Strip potential markdown code blocks if the LLM adds them
        raw_text = response.output_text.replace("```json", "").replace("```", "").strip()
        new_facts = json.loads(raw_text)
        if isinstance(new_facts, list):
            save_memory_notes(new_facts)
    except Exception as e:
        print(f"Memory extraction failed: {e}")


In [ ]:
`

### Implementing LTM using a SQlite DB

In [5]:
# Step 1: Initialize the Database (agent/memory.py)
# Instead of checking if a JSON file exists, we now "connect" to a database file. 
# If it doesn't exist, SQLite creates it automatically. We then run a command to create our Table.

In [6]:
#memory.py


import sqlite3
from datetime import datetime

DB_FILE = "memory.db"

def init_db():
    """Creates the memory table if it doesn't exist."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS memories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT,
            fact TEXT UNIQUE,
            timestamp DATETIME
        )
    ''')
    conn.commit()
    conn.close()

# Run initialization immediately when the module is imported
init_db()


def save_memory_notes(new_facts):
    """Saves new facts into the SQLite database."""
    if not new_facts:
        return

    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    now = datetime.now().isoformat()
    
    for item in new_facts:
        # 'INSERT OR REPLACE' prevents duplicate facts from clogging the DB
        cursor.execute('''
            INSERT OR REPLACE INTO memories (category, fact, timestamp)
            VALUES (?, ?, ?)
        ''', (item.get('category', 'General'), item.get('fact'), now))
    
    conn.commit()
    count = len(new_facts)
    conn.close()
    print(f"--- Database: Stored {count} facts ---")


def load_long_term_memory():
    """Fetches all memories and formats them for the System Message."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    # We pull the most recent facts first
    cursor.execute('SELECT fact, category FROM memories ORDER BY timestamp DESC')
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return ""

    formatted_memories = "\n\nRELEVANT FACTS FROM PREVIOUS SESSIONS:\n"
    for fact, category in rows:
        formatted_memories += f"- {fact} (Category: {category})\n"
    
    return formatted_memories



In [ ]:
#core.py

In [ ]:
import json
from config import client, MAX_MESSAGES
from tools.registry import tools, AVAILABLE_TOOLS
from agent.logger import log_step, print_conversation
from agent.memory import save_memory_notes



SYSTEM_MESSAGE = {
    "role": "system",
    "content": (
        "You are a versatile research assistant with access to real-time tools.\n\n"
        "1. EXA SEARCH: Use this for web-based queries. Cite source URLs.\n"
        "2. WEATHER DATA: Use this for current conditions using coordinates.\n\n"
        "Choose the most appropriate tool. You may use multiple in sequence."
    ),
}


def execute_tool_calls(tool_call):
    f_name = tool_call.name
    f_args = json.loads(tool_call.arguments)
    function_to_call = AVAILABLE_TOOLS.get(f_name)

    if function_to_call:
        results = function_to_call(**f_args)
        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(results)
        }
    raise ValueError(f"Unknown tool: {f_name}")




def run_agent(user_query, conversation):

    conversation.append({"role": "user", "content": user_query})

    # SLIDING WINDOW LOGIC
    while len(conversation) > MAX_MESSAGES:
        if len(conversation) > 1:
            conversation.pop(1)
        

    iteration_count = 0
    total_tools_executed = 0

    while True:
        iteration_count += 1
        response = client.responses.create(
            model="gpt-4o-mini",
            tools=tools,
            input=conversation,
        )

        # Append model output to conversation
        conversation += response.output

        # Collect tool calls
        tool_calls = [
            item for item in response.output 
            if item.type == "function_call"
        ]

        # NEW: Print the formatted log for this turn
        log_step(iteration_count, tool_calls)
        total_tools_executed += len(tool_calls)
        

        if not tool_calls:
            # Print Final Summary before breaking
            print(f"\n{'*'*10} AGENT FINISHED {'*'*10}")
            print(f"Total Iterations: {iteration_count}")
            print(f"Total Tool Calls: {total_tools_executed}")
            print(f"{'*'*36}\n")
            print(response.output_text)
            print_conversation(conversation)
            print(conversation)
            break

        # Execute each tool call
        for tool_call in tool_calls:
            tool_result = execute_tool_calls(tool_call)
            conversation.append(tool_result)



def extract_and_save_memories(conversation):
    """Hidden turn to extract permanent facts from the chat history."""
    extraction_prompt = (
        "Analyze the conversation above. Extract only PERMANENT facts about the user "
        "(e.g., their location, name, or specific preferences). Ignore temporary data "
        "like current weather numbers. Output ONLY a valid JSON list of objects "
        "with 'fact' and 'category' keys. Example: [{'fact': 'User lives in Lagos', 'category': 'location'}]. "
        "If no new facts, return []."
    )
    
    # Use your verbatim client.responses.create pattern
    response = client.responses.create(
        model="gpt-4o-mini",
        input=conversation + [{"role": "user", "content": extraction_prompt}]
    )
    
    try:
        # Strip potential markdown code blocks if the LLM adds them
        raw_text = response.output_text.replace("```json", "").replace("```", "").strip()
        new_facts = json.loads(raw_text)
        if isinstance(new_facts, list):
            save_memory_notes(new_facts)
    except Exception as e:
        print(f"Memory extraction failed: {e}")


In [ ]:
#main.py

from agent.core import run_agent, SYSTEM_MESSAGE, extract_and_save_memories
from agent.memory import load_long_term_memory

def main():
    # 1. LOAD: Fetch old memories and inject into System Message
    memories = load_long_term_memory()
    SYSTEM_MESSAGE["content"] += memories
    
    # 2. INITIALIZE: The session history (Short-Term Memory)
    session_history = [SYSTEM_MESSAGE]
    
    print("--- AI Research Agent (with LTM) Active ---")
    
    while True:
        try:
            user_query = input("\nWhat do you want to search for? (type 'exit' to stop): ")
            
            if user_query.lower() in ["exit", "quit"]:
                # 3. SAVE: Before quitting, summarize the session
                print("Reflecting on our conversation...")
                extract_and_save_memories(session_history)
                print("Goodbye!")
                break
                
            if not user_query.strip():
                continue
                
            # Pass the persistent session_history for Short-Term Memory
            run_agent(user_query, session_history)
            
        except Exception as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()


## Upgrading the LTM to contain RAG, Episodic and Procedural Memories

In [10]:
!pip install numpy



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_embedding(text, model="text-embedding-3-small"):
    """
    Turns a string into a list of 1,536 numbers (a Vector).
    'text-embedding-3-small' is cheap, fast, and very accurate.
    """
    # Clean the text (remove newlines which can mess up embeddings)
    text = text.replace("\n", " ")
    
    response = client.embeddings.create(
        input=[text],
        model=model
    )
    
    # Return the list of floats (the actual vector)
    return response.data[0].embedding

# Quick test if run directly
if __name__ == "__main__":
    test_text = "I love building AI agents"
    vector = get_embedding(test_text)
    print(f"Vector length: {len(vector)}")
    print(f"First 5 numbers: {vector[:5]}")


Vector length: 1536
First 5 numbers: [-0.015585217624902725, -0.037549372762441635, 0.01467990968376398, 0.0066366009414196014, -0.002318980172276497]
